# 3장 2강: 규제 — L1(Lasso), L2(Ridge), ElasticNet

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 Ridge, Lasso, ElasticNet 규제가 회귀계수와 일반화 성능에 어떤 영향을 주는지 확인합니다.

- 규제가 계수의 크기를 줄이는 이유를 이해합니다.
- `alpha` 변화에 따라 Ridge의 계수 크기와 Test 성능이 어떻게 달라지는지 확인합니다.
- Lasso가 일부 계수를 정확히 0으로 만드는지 확인합니다.
- ElasticNet이 L1과 L2를 함께 사용하는 모델이라는 점을 확인합니다.
- Ridge / Lasso / ElasticNet의 계수와 Test 성능을 비교합니다.

> 이 실습에서는 3장 2강 교안에서 다룬 규제, alpha, Lasso, Ridge, ElasticNet, StandardScaler, R² 비교만 사용합니다.

In [2]:
# ridge(릿지) : 선형 회귀에 L2규제를 적용하여 계수 크기를 제한하는 모델
# → 여러 특성중 특정 특성에 편향된 계수가 있다면 그 계수의 크기를 줄이겠다.

# L2 규제 : 회귀 계수의 제곱을 모두 더한 값을 패널티로 사용하는 규제
# → 절댓값이 큰 계수일수록 회귀계수가 굉장히 빠르게 커지기 때문에 계수를 억제하겠다.

# **다중공선성 : 특정 특성사이에 선형관계가 있어서 회귀계수 추정이 불안정해질 수 있다.
# → 같은 기간에 총 구매금액과 월평균 구매금액을 함께 사용한다면 비슷한 정보를 담고있기 때문에
# → 다중공선성 문제를 일으킬 수 있다.

# Lasso(라쏘) : 선형 회귀에 L1규제를 적용하여 계수 크기를 제한하는 모델
# → 여러 특성중 특정 특성에 편향된 계수가 있다면 그 계수의 크기를 줄이겠다.

# L1 규제 :  회귀 계수의 절댓값을 모두 더한 값을 패널티로 사용하는 규제
# → 절댓값이 큰 계수일수록 규제 강도를 높여서 일반적인 계수와 맞추려고 억제하겠다.

# ElasticNet : L1 규제와 L2 규제를 함께 적용하는 선형 회귀 모델
# → 여러 특성중 특정 특성에 편향된 계수가 있다면 그 계수의 크기를 줄이겠다.


## 실습 환경 / 데이터

- Python
- pandas
- numpy
- matplotlib
- scikit-learn
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

규제에 따른 계수 변화를 비교하기 위해 결측치가 없는 숫자형 특성 중 다음 12개를 사용합니다.

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

규제는 계수의 크기를 기준으로 벌점을 주기 때문에 특성의 스케일을 맞추는 과정이 중요합니다.

이번 실습에서는 다음 순서로 데이터를 준비합니다.

1. Ames Housing 데이터 불러오기
2. 사용할 특성과 `SalePrice` 선택
3. Train / Test 분리
4. `StandardScaler`를 이용한 특성 표준화

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train 크기:", X_train_scaled.shape)
print("Test 크기:", X_test_scaled.shape)

Train 크기: (1168, 12)
Test 크기: (292, 12)


---

# 필수 1. Ridge와 alpha에 따른 계수 축소

## 배경

Ridge는 **L2 규제**를 사용합니다.

Ridge는 계수의 제곱합에 벌점을 주기 때문에 큰 계수를 더 강하게 억제하고, 모든 계수를 0에 가깝게 줄이는 특징이 있습니다.

## 문제 1. 여러 alpha 값에서 Ridge를 비교하세요.

### 요구사항

다음 alpha 값을 사용합니다.

```python
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
```

각 alpha에 대해 다음을 수행합니다.

1. `Ridge(alpha=alpha)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test 데이터를 예측합니다.
4. Test R²를 계산합니다.
5. `abs(model.coef_).sum()`으로 전체 계수 크기를 확인합니다.
6. 결과를 DataFrame으로 정리합니다.

### 확인 포인트

- alpha가 커질수록 규제 강도가 강해지는가?
- alpha가 커질수록 전체 계수 크기가 작아지는가?
- alpha가 너무 커지면 Test 성능이 낮아질 수 있는가?

In [4]:
# TODO
# 여러 alpha 값에 대해 Ridge의 Test R²와 계수 크기를 비교하세요.

alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]

ridge_rows = []

for alpha in alpha_values:
    # 1. 모델 만들기
    model = Ridge(alpha=alpha)

    # 2. Train 데이터로 학습 (스케일링된 데이터 사용)
    model.fit(X_train_scaled, y_train)

    # 3. Test 데이터 예측
    y_pred = model.predict(X_test_scaled)

    # 4~5. Test R²와 전체 계수 크기
    ridge_rows.append({
        "alpha": alpha,
        "test_r2": r2_score(y_test, y_pred),
        "coef_abs_sum": abs(model.coef_).sum()
    })

# 6. DataFrame으로 정리
ridge_result = pd.DataFrame(ridge_rows)
display(ridge_result.round(4))

,alpha,test_r2,coef_abs_sum
0,0.001,0.7994,102172.5908
1,0.010,0.7994,102172.4637
2,0.100,0.7994,102171.0968
3,1.000,0.7993,102149.8317
4,10.000,0.7991,101725.4315
5,100.000,0.7956,97390.1202


| alpha | test_r2 | 계수 절대값 합 | 내용 |
|---:|---:|---:|---|
| 0.001 | 0.7994 | 102172.5908 | 규제가 가장 약한 설정의 값 |
| 0.010 | 0.7994 | 102172.4637 | 앞서 규제가 가장 약했던 설정의 값과 계수가 일치 |
| 0.100 | 0.7994 | 102171.0968 | 규제는 지속적으로 커ㄷ해졌지만 여전히 가장 약했던 설정의 계수와 거의 일치 |
| 1.000 | 0.7993 | 102149.8317 | 계수의 합이 조금 감소했으며 r2값이 낮아짐 |
| 10.000 | 0.7991 | 101725.4315 | 계수 축소가 뚜렷하며, r2이 낮아짐 |
| 100.000 | 0.7956 | 97390.1202 | 이번 후보중 계수의 합이 가장 낮으며 r2값도 가장 낮음 |

-> 규제를 강화하면서 계수의 전체 크기가 점차 감소했으나 test의 성능이 나아지지 않았다.  


### Q1. alpha가 커질수록 Ridge 계수는 일반적으로 어떻게 변하나요?

**답변:**  
-> 일반적으로 계수의 절대값이 작아진다.  
ridge는 모든 계수를 한 번에 0으로 바꾸지 않고 0에 가까운 방향으로 축소하려고 한다.


### Q2. alpha가 너무 크면 어떤 문제가 발생할 수 있나요?

**답변:**  
-> 중요한 특성의 계수를 지나치게 줄여서 학습에 충분한 힌트를 얻지 못할 수 있다.  
-> 즉, 과소적합이 발생할 수 있다.  

---

# 필수 2. Lasso와 ElasticNet의 계수 변화

## 문제 1. Lasso에서 0이 되는 계수를 확인하세요.

### 배경

Lasso는 **L1 규제**를 사용하며, 중요하지 않은 특성의 계수를 정확히 0으로 만들 수 있습니다.

이번에는 규제 강도를 충분히 높여 Lasso의 변수 선택 효과를 직접 확인합니다.

### 요구사항

1. `Lasso(alpha=1000, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. 각 특성의 계수를 DataFrame으로 만듭니다.
4. 계수가 0인 특성의 개수를 계산합니다.
5. Test R²를 계산합니다.

### 확인 포인트

- 일부 계수가 정확히 0이 되는가?
- Ridge와 달리 Lasso가 변수 선택 효과를 갖는 이유를 이해했는가?

In [5]:
# TODO
# Lasso(alpha=1000)를 학습하고 계수와 0 계수 개수를 확인하세요.
# 1~2. Lasso 모델 만들고 학습
lasso = Lasso(alpha=1000, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

# 3. 특성별 계수 DataFrame (절댓값이 큰 순서로 정렬)
lasso_coef = pd.DataFrame({
    "feature": features,
    "coef": lasso.coef_
}).sort_values("coef", key=abs, ascending=False)
display(lasso_coef.round(2))

# 4. 계수가 0인 특성 개수
zero_count = (lasso.coef_ == 0).sum()
print(f"계수가 0인 특성 수: {zero_count}개 / 전체 {len(features)}개")
print("계수가 0인 특성:", lasso_coef.loc[lasso_coef["coef"] == 0, "feature"].tolist())

# 5. Test R²
lasso_r2 = r2_score(y_test, lasso.predict(X_test_scaled))
print(f"Lasso Test R²: {lasso_r2:.4f}")


,feature,coef
0,OverallQual,25520.69
1,GrLivArea,21419.63
4,TotalBsmtSF,7305.57
9,YearBuilt,6966.92
2,GarageCars,6820.21
10,YearRemodAdd,6418.69
11,Fireplaces,6221.90
3,GarageArea,4370.09
5,1stFlrSF,3521.10
8,TotRmsAbvGrd,83.20


계수가 0인 특성 수: 2개 / 전체 12개
계수가 0인 특성: ['FullBath', '2ndFlrSF']
Lasso Test R²: 0.7960


결과 :  
OverallQual(품질)컬럼의 계수 25520의 의미 : 품질이 높을 수록 예측 가격이 높아지는 방향, 그 강도  
GrLivArea(차고)컬럼의 계수 21419의 의미 : 지상 면정의 정보도 예측 가격에 양의 계수로 반영되고 있다.  
2ndFlrSF(2층면적)컬럼의 계수 0.0의 의미 : 이 모델에서 이 2층 면적의 직접적인 기여도는 0이다.  
FullBath(욕실)컬럼의 계수 -0.0의 의미 : 이 모델에서 욕실은 예측 가격에 직접적인 기여도는 0이다.  

- 0의 계수가 2개인 것의 의미  
전체 12개의 특성중 2개의 특성이 0의 특징을 보임  
-> 따라서 이 두 열의 값이 바뀌거나 제거되어도 나머지 10개의 열이 같으면 모델의 결과는 달라지지 않음  

- 계수 0은 그러면 모델링할 때 삭제해도 무방하냐?  
-> 2층 면적이나 욕실수는 계수가 없어서 주택가격과 관련이 없다는 거 아니냐?  
-> **데이터를 분할, 규제강도의 변화, x의 변화가 있는 경우 계수가 0이 아닐수도 있음**  

### Q3. Lasso에서 계수가 0이 된다는 것은 어떤 의미인가요?

**답변:**  
해당 특성(x)가 최종 예측식에서 영향을 주지 않는다는 의미  
-> Lasso의 특징 : 중요도가 낮다고 판단한 특성을 자동으로 모델의 힌트로 사용하지 않음 (제거하는 효과)

## 문제 2. ElasticNet을 학습하고 L1/L2 혼합을 확인하세요.

### 배경

ElasticNet은 L1과 L2 규제를 함께 사용합니다.

`l1_ratio`는 두 규제의 비율을 조절합니다.

- `l1_ratio`가 1에 가까움 → Lasso 성격이 강함
- `l1_ratio`가 0에 가까움 → Ridge 성격이 강함

### 요구사항

1. `ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)` 모델을 만듭니다.
2. Train 데이터로 학습합니다.
3. Test R²를 계산합니다.
4. 각 특성의 계수를 확인합니다.

### 확인 포인트

- `alpha`와 `l1_ratio`의 역할을 구분할 수 있는가?
- ElasticNet이 Lasso와 Ridge의 성격을 함께 가진다는 점을 설명할 수 있는가?

In [6]:
# TODO
# ElasticNet(alpha=0.1, l1_ratio=0.5)을 학습하세요.
# 1~2. ElasticNet 모델 만들고 학습
enet = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
enet.fit(X_train_scaled, y_train)

# 3. Test R²
enet_r2 = r2_score(y_test, enet.predict(X_test_scaled))
print(f"ElasticNet Test R²: {enet_r2:.4f}")

# 4. 특성별 계수 확인
enet_coef = pd.DataFrame({
    "feature": features,
    "coef": enet.coef_
}).sort_values("coef", key=abs, ascending=False)
display(enet_coef.round(2))

print(f"계수가 0인 특성 수: {(enet.coef_ == 0).sum()}개")


ElasticNet Test R²: 0.7973


,feature,coef
0,OverallQual,23514.63
1,GrLivArea,12670.77
5,1stFlrSF,9237.82
9,YearBuilt,8072.29
4,TotalBsmtSF,8016.09
10,YearRemodAdd,7518.26
11,Fireplaces,6995.26
6,2ndFlrSF,6873.82
2,GarageCars,6785.13
3,GarageArea,5137.44


계수가 0인 특성 수: 0개


1_ratio = 0.5 : L1항의 앞에 0.05, L2 제곱합 앞에는 0.025 , r=0.5라는 의미는 두 규제의 혼합비율을 중간으로 쓰겠다는 의미  

결과  
OverallQual(품질)컬럼의 계수 23514의 의미 : 양의 계수중 가장 큰 값, 품징 정보가 가격예측에 가장 크게 반영  
GrLivArea(지상면적)컬럼의 계수 12670의 의미 : 양의 계수중 두번째로 큰 값, 면적도 가격예측에 크게 반영  
2ndFlrSF(2층면적)컬럼의 계수 6873의 의미 : Lasso에서는 0이었는데 현재에서는 가격예측에 기여  
FullBath(욕실)컬럼의 계수 -1895의 의미 : 다른 입력이 같다면 가격예측에 음의 영향을 보인다.(12컬럼 중 가장 작은 영향력을 보인다.)  

### Q4. `alpha`와 `l1_ratio`는 각각 무엇을 조절하나요?

**답변:**  
alpha : 전체 규제의 강도를 조절  
l1_ratio : 규제 안에서 L1과 L2를 어떤 비율로 사용할지 조절  
-> L1_ratio가 1에 가까워질수록 Lasso, 0에 가까워질수록 Ridge에 성형이 강해진다.  

---

# 과제 1. 다중공선성 진단과 Ridge / Lasso / ElasticNet 비교

## 배경

같은 Train/Test 데이터에서 세 규제 모델을 비교합니다. 성취도 평가항목에 맞춰, 비교 전에 **다중공선성의 계수 해석 영향과 VIF 진단·처치 선택**을 추가합니다.

다중공선성은 입력 특성들이 서로 강한 선형 관계를 갖는 상황입니다. VIF는 한 특성을 나머지 특성으로 설명할 때 구한 결정계수 R²를 이용해 `1 / (1 - R²)`로 계산합니다. 이때 예측 대상은 `SalePrice`가 아니라 **진단할 입력 특성 하나**이며, 보조 회귀에는 절편을 포함합니다.

## 요구사항

### A. 다중공선성 진단과 처치 방향 선택

1. **Train의 입력 특성만** 이용해 특성별 VIF를 계산하고 내림차순 표 `vif_df`를 출력합니다. 기존 `X_train_scaled`를 사용해도 됩니다. 표준화는 특성 간 선형 관계 자체를 제거하지 않습니다.
2. 이번 과제에서는 `VIF < 5`: 상대적으로 낮음, `5 ≤ VIF < 10`: 주의, `VIF ≥ 10`: 높은 수준으로 분류합니다. 이는 절대적 삭제 규칙이 아닌 실습용 판단 기준입니다.
3. 주의·높은 수준인 특성을 특성명과 VIF로 제시합니다. 없으면 없다고 작성합니다. 완전한 선형 중복으로 VIF가 무한대인 경우 높은 수준으로 해석합니다. 상수 특성의 VIF는 정의하기 어려우므로 별도로 표시합니다.
4. 다중공선성이 계수의 크기·부호·안정성과 개별 변수 해석에 미치는 영향을 설명합니다.
5. 특성의 의미와 VIF 결과를 근거로 **변수 제거·결합·규제 중 한 가지 처치 방향**을 선택하고 대상·이유·한계를 설명합니다. 높은 VIF가 없으면 불필요한 제거를 하지 않는 이유와 기존 규제 비교의 목적을 설명해도 됩니다. 제거·결합의 추가 구현은 요구하지 않습니다.

### B. 기존 규제 모델 비교

다음 설정을 그대로 사용합니다.

```python
Ridge(alpha=1.0)
Lasso(alpha=1000, max_iter=10000)
ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
```

1. 세 모델을 동일한 표준화 Train 데이터로 학습합니다.
2. 동일한 Test 데이터의 R²를 계산합니다.
3. 전체 계수 크기 `abs(coef_).sum()`과 계수가 정확히 0인 개수를 계산합니다.
4. 결과를 `comparison_df`로 정리하고, 특성별 계수를 `task_coef_df`로 출력합니다.
5. Test R²와 계수 처리 차이를 함께 설명하고 Q5~Q9에 답합니다.

> Test 점수는 지정된 세 설정의 결과를 비교·해석하는 데 사용합니다. 이 점수를 보고 변수를 제거하거나 alpha를 반복 조정하지 않습니다. 실제 모델 선택·튜닝에는 Train 내부 검증 또는 CV가 필요합니다.

### 확인 포인트

- Train 입력 특성만 사용하여 VIF를 계산했는가?
- 기준과 실제 수치를 근거로 주의할 특성을 판단했는가?
- 다중공선성의 계수 해석 영향과 처치 선택 이유를 설명했는가?
- 규제가 입력 특성의 VIF 자체를 낮추는 것은 아님을 구분했는가?
- 같은 데이터에서 세 모델의 Test R²·계수 크기·0 계수 개수를 비교했는가?


In [8]:
# TODO
# A. Train 특성별 VIF 계산 및 판단
from sklearn.linear_model import LinearRegression

# Train 입력 특성만 사용 (SalePrice, Test는 사용하지 않음)
X_vif = pd.DataFrame(X_train_scaled, columns=features)

vif_rows = []
for col in features:
    # 상수 특성은 VIF를 정의하기 어려우므로 따로 표시
    if X_train[col].nunique() <= 1:
        vif_rows.append({"feature": col, "aux_R2": np.nan, "VIF": np.nan})
        continue

    others = [c for c in features if c != col]

    # 진단할 특성 하나를 나머지 특성으로 설명하는 보조 회귀 (절편 포함)
    aux = LinearRegression()
    aux.fit(X_vif[others], X_vif[col])
    r2 = aux.score(X_vif[others], X_vif[col])

    # 완전히 설명되면(R² = 1) VIF는 무한대
    vif = np.inf if r2 >= 1 - 1e-12 else 1 / (1 - r2)
    vif_rows.append({"feature": col, "aux_R2": r2, "VIF": vif})

vif_df = pd.DataFrame(vif_rows)

def vif_level(v):
    if pd.isna(v):
        return "상수 특성(정의 어려움)"
    if v >= 10:
        return "높은 수준"
    if v >= 5:
        return "주의"
    return "상대적으로 낮음"

vif_df["level"] = vif_df["VIF"].apply(vif_level)
vif_df = vif_df.sort_values("VIF", ascending=False).reset_index(drop=True)
display(vif_df.round(3))

flagged = vif_df[vif_df["VIF"] >= 5]
print("주의·높은 수준 특성:")
print(flagged[["feature", "VIF", "level"]].round(2).to_string(index=False) if len(flagged) > 0 else "없음")

# B. 기존 세 규제 모델 비교 및 특성별 계수 표 출력
models = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1000, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000),
}

comparison_rows = []
coef_dict = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)          # 같은 표준화 Train
    y_pred = model.predict(X_test_scaled)       # 같은 Test

    comparison_rows.append({
        "model": name,
        "test_r2": r2_score(y_test, y_pred),
        "coef_abs_sum": abs(model.coef_).sum(),
        "zero_coef_count": int((model.coef_ == 0).sum())
    })
    coef_dict[name] = model.coef_

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.round(4))

task_coef_df = pd.DataFrame(coef_dict, index=features)
display(task_coef_df.round(2))


,feature,aux_R2,VIF,level
0,GrLivArea,0.992,128.777,높은 수준
1,2ndFlrSF,0.989,91.621,높은 수준
2,1stFlrSF,0.986,73.586,높은 수준
3,GarageCars,0.811,5.291,주의
4,GarageArea,0.801,5.032,주의
5,TotalBsmtSF,0.736,3.793,상대적으로 낮음
6,TotRmsAbvGrd,0.700,3.333,상대적으로 낮음
7,OverallQual,0.628,2.689,상대적으로 낮음
8,YearBuilt,0.581,2.389,상대적으로 낮음
9,FullBath,0.552,2.234,상대적으로 낮음


주의·높은 수준 특성:
   feature    VIF level
 GrLivArea 128.78 높은 수준
  2ndFlrSF  91.62 높은 수준
  1stFlrSF  73.59 높은 수준
GarageCars   5.29    주의
GarageArea   5.03    주의


,model,test_r2,coef_abs_sum,zero_coef_count
0,Ridge,0.7993,102149.8317,0
1,Lasso,0.7960,88648.0061,2
2,ElasticNet,0.7973,99258.8443,0


,Ridge,Lasso,ElasticNet
OverallQual,25310.89,25520.69,23514.63
GrLivArea,13960.71,21419.63,12670.77
GarageCars,7041.54,6820.21,6785.13
GarageArea,4382.96,4370.09,5137.44
TotalBsmtSF,7238.59,7305.57,8016.09
1stFlrSF,10011.02,3521.10,9237.82
2ndFlrSF,7098.90,0.00,6873.82
FullBath,-3154.79,-0.00,-1895.88
TotRmsAbvGrd,1602.13,83.20,2541.48
YearBuilt,8343.12,6966.92,8072.29


### Q5. Ridge와 Lasso의 가장 큰 계수 처리 차이는 무엇인가요?

**답변:**
- Ridge: L2규제를 사용하여 계수의 크기를 전반적으로 줄이지만, 정확히 0으로 만들지는 않아 12개 특성이 모두 남음 (0 계수 0개, 계수 합 102,149.83)  
- Lasso: L1규제를 사용하여 계수를 줄이고 일부 계수를 정확히 0으로 만들어, 2ndFlrSF, FullBath를 모델에서 제외함 (0 계수 2개, 계수 합 88,648.01)  
-> Ridge는 계수 축소, Lasso는 계수 축소와 함께 feature selection까지 함

### Q6. 규제 모델의 성능을 비교할 때 Test R²뿐 아니라 계수도 함께 보는 이유는 무엇인가요?

**답변:**
- 세 모델의 Test R²는 0.7960~0.7993으로 차이가 0.004 이내지만, 계수 처리 방식은 각각 다르기 때문
- Lasso : 전체 계수 크기는 줄이면서 면적 정보를 GrLivArea(21,419.63)에 몰아주고 2ndFlrSF를 0으로 만듦  
- Ridge, ElasticNet : 겹치는 면적 정보를 GrLivArea·1stFlrSF·2ndFlrSF에 나눠 줌  
  (Ridge 13,960.71 / 10,011.02 / 7,098.90, ElasticNet 12,670.77 / 9,237.82 / 6,873.82)  
-> 성능이 비슷해도 어떤 변수를 쓰고 어떻게 해석되는지가 다르므로, 해석 가능성·변수 선택·계수 안정성을 함께 보고 목적에 맞는 모델을 골라야 함

### Q7. 다중공선성은 회귀계수의 크기·부호·안정성과 개별 변수의 영향 해석에 어떤 문제를 일으킬 수 있나요?

**답변:**  
- 크기: 비슷한 정보를 가진 feature들이 영향력을 나눠 갖거나 서로 상쇄하면서 계수가 실제보다 작거나 크게 나타날 수 있음  
- 부호: 다른 특성이 이미 설명한 부분을 빼고 남은 차이를 맞추면서 부호가 상식과 반대로 나올 수 있음  
(FullBath 계수가 Ridge -3,154.79, ElasticNet -1,895.88로 음수)
- 안정성: 모델이 조금만 바뀌어도 계수가 크게 달라짐 (GrLivArea가 Ridge 13,960.71, Lasso 21,419.63 / 1stFlrSF가 Ridge 10,011.02, Lasso 3,521.10)  
-> 예측 성능은 유지되더라도 "이 변수가 1 늘면 SalePrice가 얼마 변한다"는 개별 변수의 영향 해석은 믿기 어려워짐

### Q8. VIF 판단 기준은 무엇이며, 계산 결과에서 주의하거나 처치를 검토할 특성은 무엇인가요? 특성명과 VIF를 근거로 답하세요.

**답변:**
- `기준: VIF < 5 상대적으로 낮음, 5 ≤ VIF < 10 주의, VIF ≥ 10 높은 수준`
- `높은 수준: GrLivArea (128.78), 2ndFlrSF (91.62), 1stFlrSF (73.59)`  
-> GrLivArea는 보조 회귀 R²가 0.992로, 나머지 특성(1stFlrSF + 2ndFlrSF 등)들로 99% 정도 설명된다고 볼 수 있음.  
지상 거주 면적이 1층과 2층 면적의 합에 가까워 세 특성이 함께 높게 나타남  

- `주의: GarageCars (5.29), GarageArea (5.03)`  
-> GarageCars는 보조 회귀 R²가 0.811로, 나머지 특성(GarageArea 등)으로 81% 정도 설명된다고 볼 수 있음.  
차고 수용 대수와 차고 면적이 모두 차고 크기를 나타내기 때문에 두 특성이 함께 주의 수준으로 높게 나타남  
- 나머지 7개 특성은 모두 VIF 5 미만이며, VIF가 무한대이거나 상수인 특성은 없음

### Q9. VIF 결과와 특성의 의미를 고려하여 변수 제거·결합·규제 중 어떤 처치 방법을 선택하겠나요? 구체적인 대상, 선택 이유와 한계를 설명하세요.

**답변:**

- 선택: 변수 제거
- 대상: 1stFlrSF, 2ndFlrSF를 제거하고 GrLivArea를 남김
- 이유: GrLivArea가 1층과 2층 면적의 합에 가까워 두 특성의 면적 정보를 99% 정도로 설명하고 있으며,  
VIF가 70~130으로 가장 높은 문제가 이 세 특성의 중복을 가리키는 것으로 보임.  
Lasso도 2ndFlrSF를 0으로 만들어 같은 방향을 보여 줌. 차고 두 특성은 VIF가 주의 수준(5.0~5.3)이고 대수와 면적의 의미가 달라 유지함

- 한계: 1층·2층 면적 구성(단층/2층 구조)에 대한 정보가 사라지고, 제거 후 VIF를 다시 계산해 확인해야 함.  
제거가 성능에 미치는 영향은 Test가 아닌 Train 내부 CV로 확인해야 함  
-> 규제는 계수의 크기와 흔들림을 줄여 줄 뿐, 입력 특성끼리의 관계로 계산되는 VIF 자체를 낮추지는 않으므로,  
높은 VIF를 해소하려면 제거나 결합처럼 입력 특성을 바꾸는 처치가 필요할 수 있음


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 규제는 손실함수에 무엇을 추가하는 방법인가요?  
-> 손실함수에 계수 크기에 대한 벌점을 추가한다.  

2. alpha가 커지면 규제 강도는 어떻게 변하나요?  
-> 규제 강도가 강해진다.  

3. Ridge의 계수는 일반적으로 어떻게 변하나요?  
-> 모든 계수를 0에 가까운 방향으로 축소함, 단 0으로 만들지는 않음  

4. Lasso가 변수 선택 효과를 가지는 이유는 무엇인가요?  
-> 일부 계수를 0으로 만들 수 있다.  

5. ElasticNet의 `l1_ratio=0.5`는 어떤 의미인가요?  
-> L1과 L2 규제를 절반씩 섞는 설정으로 이해할 수 있다.  

6. 규제 전에 특성 스케일을 맞추는 이유는 무엇인가요?  
-> 규제는 계수 크기를 기준으로 패널티를 부여하기 때문에 특성들의 값의 범위가 다르면 패널티가 다르게 반영될 수 있음  
-> 따라서 특성 스케일을 맞춘뒤 규제를 적용한다.